<a href="https://colab.research.google.com/github/kausarfatima2626/covid-19-detection-from-xray-images-using-cnn/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Step 1: Install kagglehub and download dataset automatically
!pip install kagglehub -q

import kagglehub
import os
import shutil

print("Downloading dataset from Kaggle...")
# Download chest x-ray dataset for covid19 & pneumonia
path = kagglehub.dataset_download("prashant268/chest-xray-covid19-pneumonia")

print("Path to downloaded files:", path)

# Create local dataset folder in Colab
os.makedirs("dataset", exist_ok=True)

# List contents to confirm
print("Downloaded dataset contents:")
print(os.listdir(path))

100%|██████████| 2.06G/2.06G [01:50<00:00, 19.9MB/s]

Extracting files...


Path to downloaded files: /root/.cache/kagglehub/datasets/prashant268/chest-xray-covid19-pneumonia/versions/2
Downloaded dataset contents:
['Data']


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Dataset ka main path
dataset_dir = os.path.join(path, 'Data')

# Image parameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Data Augmentation & Normalization
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2, # 80% Training, 20% Validation
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)

# Training Data Generator
train_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Validation Data Generator
val_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print("\nClass Labels/Mapping:", train_generator.class_indices)

Found 5147 images belonging to 2 classes.
Found 1285 images belonging to 2 classes.

Class Labels/Mapping: {'test': 0, 'train': 1}


In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Train folder ka exact path
train_dir = os.path.join(path, 'Data', 'train')
test_dir = os.path.join(path, 'Data', 'test')

# Image parameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Data Augmentation & Normalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Training Data Generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Validation Data Generator
val_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("\nCorrected Class Labels/Mapping:", train_generator.class_indices)


Found 5144 images belonging to 3 classes.
Found 1288 images belonging to 3 classes.

Corrected Class Labels/Mapping: {'COVID19': 0, 'NORMAL': 1, 'PNEUMONIA': 2}


In [9]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# Step 1: Pre-trained Base Model (MobileNetV2) load karna
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Base model ke layers ko freeze karna
base_model.trainable = False

# Step 2: Apne custom layers add karna
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(3, activation='softmax')(x) # 3 classes: COVID19, NORMAL, PNEUMONIA

model = Model(inputs=base_model.input, outputs=predictions)

# Step 3: Model compile karna
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Step 4: Model train karna (5 Epochs)
print("Starting Model Training...")
history = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)

# Step 5: Trained Model ko save karna
model.save("model.h5")
print("\nModel saved successfully as model.h5!")

Starting Model Training...
Epoch 1/5
161/161 ━━━━━━━━━━━━━━━━━━━━ 233s 1s/step - accuracy: 0.8820 - loss: 0.3032 - val_accuracy: 0.9084 - val_loss: 0.2274
Epoch 2/5
161/161 ━━━━━━━━━━━━━━━━━━━━ 232s 1s/step - accuracy: 0.9306 - loss: 0.1724 - val_accuracy: 0.9177 - val_loss: 0.2053
Epoch 3/5
161/161 ━━━━━━━━━━━━━━━━━━━━ 231s 1s/step - accuracy: 0.9372 - loss: 0.1600 - val_accuracy: 0.9363 - val_loss: 0.1675
Epoch 4/5
161/161 ━━━━━━━━━━━━━━━━━━━━ 231s 1s/step - accuracy: 0.9425 - loss: 0.1557 - val_accuracy: 0.9286 - val_loss: 0.1775
Epoch 5/5
161/161 ━━━━━━━━━━━━━━━━━━━━ 213s 1s/step - accuracy: 0.9479 - loss: 0.1387 - val_accuracy: 0.9068 - val_loss: 0.2281



Model saved successfully as model.h5!
